# Name Machine - Data analysis

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

FILES = {
    "national": "names.csv",
    "regional": "name_region.csv",
    "departemental": "name_dept.csv",
}



## Chargement et inventaire

In [2]:
data = {
    level: pd.read_csv(
        path, sep=";", encoding="utf-8",
        keep_default_na=False, na_values=[""],
        dtype={"prenom": "string", "reg": "string", "dpt": "string"},
    )
    for level, path in FILES.items()
}

inventory = pd.DataFrame([
    {
        "niveau": level,
        "lignes": len(df),
        "colonnes": len(df.columns),
        "prénoms_distincts": df["prenom"].nunique(),
        "année_min": df["periode"].min(),
        "année_max": df["periode"].max(),
        "années": df["periode"].nunique(),
        "zones": df["reg" if level == "regional" else "dpt"].nunique()
                 if level != "national" else None,
    }
    for level, df in data.items()
])
display(inventory)
for level, df in data.items():
    print(f"\n{level}: {list(df.columns)}")
    display(df.head(3))


,niveau,lignes,colonnes,prénoms_distincts,année_min,année_max,années,zones
0,national,724645,5,49163,1900,2025,126,NaN
1,regional,1928764,6,27213,1900,2025,126,18.0
2,departemental,3969540,6,17671,1900,2025,126,101.0



national: ['sexe', 'prenom', 'periode', 'valeur', 'rang']


,sexe,prenom,periode,valeur,rang
0,1,GABRIEL,2025,4625,1
1,1,NOAH,2025,3465,2
2,1,LÉO,2025,3420,3



regional: ['sexe', 'prenom', 'periode', 'reg', 'valeur', 'rang']


,sexe,prenom,periode,reg,valeur,rang
0,1,NOAH,2025,01,20,1
1,1,JAYDEN,2025,01,20,2
2,1,LIAM,2025,01,15,3



departemental: ['sexe', 'prenom', 'periode', 'dpt', 'valeur', 'rang']


,sexe,prenom,periode,dpt,valeur,rang
0,1,GABRIEL,2025,01,30,1
1,1,LOUIS,2025,01,20,2
2,1,MAËL,2025,01,20,3


## Contrôles de qualité et d’unicité

In [ ]:
KEYS = {
    "national": ["sexe", "prenom", "periode"],
    "regional": ["sexe", "prenom", "periode", "reg"],
    "departemental": ["sexe", "prenom", "periode", "dpt"],
}

quality = pd.DataFrame([
    {
        "niveau": level,
        "doublons_clés": df.duplicated(subset=KEYS[level]).sum(),
        "prénoms_manquants": df["prenom"].isna().sum(),
        "autres_valeurs_manquantes": df.drop(columns=["prenom"]).isna().sum().sum(),
        "valeurs_non_positives": (df["valeur"] <= 0).sum(),
        "rangs_non_positifs": (df["rang"] <= 0).sum(),
        "valeurs_non_multiples_de_5": (df["valeur"] % 5 != 0).sum(),
        "valeur_min": df["valeur"].min(),
        "valeur_max": df["valeur"].max(),
    }
    for level, df in data.items()
])
display(quality)
display(data["national"].loc[data["national"]["prenom"].isna()])
assert quality["doublons_clés"].eq(0).all()
assert quality["autres_valeurs_manquantes"].eq(0).all()


## Couverture temporelle par territoire

In [ ]:
coverage = []
for level, column in (("regional", "reg"), ("departemental", "dpt")):
    series = data[level].groupby(column)["periode"].agg(["min", "max", "nunique"])
    incomplete = series.loc[series["nunique"] < 126].reset_index()
    incomplete.insert(0, "niveau", level)
    coverage.append(incomplete)
coverage_table = pd.concat(coverage, ignore_index=True)
display(coverage_table)


## Comparabilité des effectifs

In [ ]:
annual_totals = pd.concat([
    df.groupby("periode")["valeur"].sum().rename(level)
    for level, df in data.items()
], axis=1).sort_index()
annual_totals.index.name = "année"
display(annual_totals.loc[[1900, 1950, 2000, 2025]])

comparison_2025 = annual_totals.loc[2025].to_frame("effectifs_publiés")
comparison_2025["écart_au_national_pct"] = (
    100
    * (
        comparison_2025["effectifs_publiés"]
        / comparison_2025.loc["national", "effectifs_publiés"]
        - 1
    )
).round(2)
display(comparison_2025)


## Classement national 2025

In [ ]:
top_2025 = (
    data["national"].query("periode == 2025")
    .sort_values(["sexe", "rang"])
    .groupby("sexe", group_keys=False).head(5)
    [["sexe", "prenom", "periode", "rang", "valeur"]]
)
display(top_2025)


## Visualisation de contrôle

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
annual_totals.plot(ax=ax, linewidth=1.6)
ax.set_title("Somme annuelle des effectifs publiés par niveau")
ax.set_xlabel("Année")
ax.set_ylabel("Somme des valeurs publiées")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Vérifications finales

In [ ]:
assert len(data["national"]) == 724645
assert len(data["regional"]) == 1928764
assert len(data["departemental"]) == 3969540
assert annual_totals.loc[2025, "national"] == 589485
assert annual_totals.loc[2025, "regional"] == 528105
assert annual_totals.loc[2025, "departemental"] == 440630
assert data["national"]["prenom"].isna().sum() == 0
assert data["national"]["prenom"].eq("NA").any()
